In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/content/train.txt', sep = ';', header = None,
names = ['text', 'emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,0


In [5]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0;
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i += 1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [6]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [7]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [8]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','', string.punctuation))

In [9]:
df['text'].apply(remove_punc)

,text
0,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy
...,...
15995,i just had a very brief time in the beanbag an...
15996,i am now turning and i feel pathetic that i am...
15997,i feel strong and good overall
15998,i feel like this was such a rude comment and i...


In [10]:
def remove_nums(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new += i
  return new

df['text'] = df['text'].apply(remove_nums)

In [11]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new

df['text'] = df['text'].apply(remove_emojis)

In [12]:
import nltk

In [13]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [14]:
stop_words = set(stopwords.words('english'))

In [15]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [16]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [17]:
nltk.download('punkt_tab', quiet=True)
def remove(txt):
  words = word_tokenize(txt)
  new = ""
  for word in words:
    if word not in stop_words:
      new += word + " "
  return new

df['text'] = df['text'].apply(remove)

In [18]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [19]:
from sklearn.model_selection import train_test_split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.33, random_state=42)

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [25]:
bow_vectorizer = CountVectorizer()

In [29]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [30]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 96931 stored elements and shape (10720, 12145)>

In [32]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [33]:
nb_model = MultinomialNB()

In [34]:
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [35]:
pred_nb = nb_model.predict(X_test_bow)

In [37]:
print(accuracy_score(y_test, pred_nb))

0.7649621212121213


In [38]:
tfidf = TfidfVectorizer()

In [39]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [40]:
nb2_model = MultinomialNB()

In [41]:
nb2_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [42]:
pred_nb2 = nb2_model.predict(X_test_tfidf)

In [44]:
print(accuracy_score(y_test, pred_nb2))

0.6609848484848485


In [45]:
from sklearn.linear_model import LogisticRegression

In [47]:
logistic_reg = LogisticRegression(max_iter = 1000)

In [48]:
logistic_reg.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [49]:
pred_logistic = logistic_reg.predict(X_test_tfidf)

In [50]:
print(accuracy_score(y_test, pred_logistic))

0.8473484848484848
